# Improved Solution - Datathon 2026

Notebook ini memberikan solusi lengkap dengan:
1. Advanced feature engineering
2. SMOTE untuk handle class imbalance
3. Multiple model alternatives
4. Ensemble methods
5. Extensive hyperparameter tuning
6. Feature selection
7. Submission generation

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectKBest, f_classif, RFE
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from imblearn.over_sampling import SMOTE
from scipy.stats import uniform, randint
import warnings
warnings.filterwarnings('ignore')
from google.colab import drive
drive.mount('/content/drive')

# Install CatBoost if not already installed, and import it right after
try:
    import catboost
except ImportError:
    !pip install catboost
    import catboost # Import again after installation

# Now import CatBoostClassifier after ensuring catboost is installed
from catboost import CatBoostClassifier

# Load data
train = pd.read_csv('/content/drive/MyDrive/Colab_Data/data/train.csv')
test = pd.read_csv('/content/drive/MyDrive/Colab_Data/data/test.csv')
sample_sub = pd.read_csv('/content/drive/MyDrive/Colab_Data/data/sample_submission.csv')

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")
print(f"\nTarget distribution:")
print(train['target'].value_counts())

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.3 MB/s eta 0:00:00
Train shape: (3200, 43)
Test shape: (800, 42)

Target distribution:
target
0    813
3    807
1    796
2    784
Name: count, dtype: int64


## 1. Advanced Feature Engineering

In [3]:
def create_features(df):
    """
    Buat fitur tambahan dari data mentah dengan advanced feature engineering
    """
    df = df.copy()

    # 1. Statistik nilai mingguan
    nilai_cols = [col for col in df.columns if 'nilai_minggu' in col]
    df['nilai_mean'] = df[nilai_cols].mean(axis=1)
    df['nilai_std'] = df[nilai_cols].std(axis=1)
    df['nilai_max'] = df[nilai_cols].max(axis=1)
    df['nilai_min'] = df[nilai_cols].min(axis=1)
    df['nilai_range'] = df['nilai_max'] - df['nilai_min']

    # 2. Trend nilai (naik/turun)
    df['nilai_trend'] = df[nilai_cols].iloc[:, -1] - df[nilai_cols].iloc[:, 0]

    # 3. Advanced Rolling Statistics
    df['nilai_rolling_mean_3'] = df[nilai_cols].iloc[:, -3:].mean(axis=1)
    df['nilai_rolling_std_3'] = df[nilai_cols].iloc[:, -3:].std(axis=1)
    df['nilai_slope'] = df[nilai_cols].iloc[:, -1] - df[nilai_cols].iloc[:, 0]
    df['nilai_volatility'] = df[nilai_cols].std(axis=1)
    df['nilai_75th'] = df[nilai_cols].quantile(0.75, axis=1)
    df['nilai_25th'] = df[nilai_cols].quantile(0.25, axis=1)
    df['nilai_iqr'] = df['nilai_75th'] - df['nilai_25th']

    # 4. Statistik aktivitas harian
    aktivitas_cols = [col for col in df.columns if 'aktivitas_hari' in col]
    df['aktivitas_mean'] = df[aktivitas_cols].mean(axis=1)
    df['aktivitas_std'] = df[aktivitas_cols].std(axis=1)
    df['aktivitas_max'] = df[aktivitas_cols].max(axis=1)
    df['aktivitas_min'] = df[aktivitas_cols].min(axis=1)
    df['aktivitas_range'] = df['aktivitas_max'] - df['aktivitas_min']

    # Rolling statistics for aktivitas
    df['aktivitas_rolling_mean_3'] = df[aktivitas_cols].iloc[:, -3:].mean(axis=1)
    df['aktivitas_rolling_std_3'] = df[aktivitas_cols].iloc[:, -3:].std(axis=1)

    # 5. Rasio tugas
    df['rasio_tugas'] = df['tugas_selesai'] / df['tugas_diberikan']
    df['rasio_tugas'] = df['rasio_tugas'].fillna(0)

    # 6. Kombinasi fitur skor
    df['total_skor'] = df['skor_motivasi'] + df['skor_kedisiplinan'] + df['skor_tryout'] + df['skor_literasi']
    df['motivasi_kedisiplinan'] = df['skor_motivasi'] * df['skor_kedisiplinan']
    df['motivasi_tryout'] = df['skor_motivasi'] * df['skor_tryout']
    df['kedisiplinan_literasi'] = df['skor_kedisiplinan'] * df['skor_literasi']

    # 7. Fitur interaksi lain
    df['kehadiran_minat'] = df['indeks_kehadiran'] * df['skor_minat_belajar']
    df['tryout_ekstra'] = df['skor_tryout'] * df['skor_ekstrakurikuler']
    df['motivasi_ekstra'] = df['skor_motivasi'] * df['skor_ekstrakurikuler']

    # 8. Polynomial features untuk fitur penting
    df['nilai_mean_squared'] = df['nilai_mean'] ** 2
    df['total_skor_squared'] = df['total_skor'] ** 2
    df['rasio_tugas_squared'] = df['rasio_tugas'] ** 2

    # 9. Domain-Specific Features (NEW)
    # Consistency metrics
    df['nilai_consistency'] = 1 - (df['nilai_std'] / (df['nilai_mean'].abs() + 1e-6))

    # Improvement rate
    df['nilai_improvement_rate'] = (df[nilai_cols].iloc[:, -1] - df[nilai_cols].iloc[:, 0]) / 12

    # Activity engagement score
    df['activity_engagement'] = df[aktivitas_cols].gt(df[aktivitas_cols].mean(axis=1)).sum(axis=1) / len(aktivitas_cols)

    # Performance trend (positive/negative)
    df['performance_trend'] = np.where(df['nilai_trend'] > 0, 1, 0)

    # Task completion efficiency
    df['task_efficiency'] = df['rasio_tugas'] * df['aktivitas_mean']

    # Academic potential score
    df['academic_potential'] = (
        df['skor_tryout'] * 0.4 +
        df['total_skor'] * 0.3 +
        df['nilai_mean'] * 0.3
    )

    # Motivation consistency
    df['motivation_consistency'] = df['skor_motivasi'] * df['nilai_consistency']

    # Learning momentum
    df['learning_momentum'] = df['nilai_rolling_mean_3'] * df['aktivitas_rolling_mean_3']

    # 10. ADVANCED FEATURES - NEW
    # Percentile-based features
    df['nilai_percentile_90'] = df[nilai_cols].quantile(0.9, axis=1)
    df['nilai_percentile_10'] = df[nilai_cols].quantile(0.1, axis=1)
    df['aktivitas_percentile_90'] = df[aktivitas_cols].quantile(0.9, axis=1)
    df['aktivitas_percentile_10'] = df[aktivitas_cols].quantile(0.1, axis=1)

    # Skewness and kurtosis
    df['nilai_skew'] = df[nilai_cols].skew(axis=1)
    df['nilai_kurtosis'] = df[nilai_cols].kurtosis(axis=1)
    df['aktivitas_skew'] = df[aktivitas_cols].skew(axis=1)
    df['aktivitas_kurtosis'] = df[aktivitas_cols].kurtosis(axis=1)

    # Momentum indicators
    df['nilai_momentum_5'] = df[nilai_cols].iloc[:, -1] - df[nilai_cols].iloc[:, -5]
    df['aktivitas_momentum_5'] = df[aktivitas_cols].iloc[:, -1] - df[aktivitas_cols].iloc[:, -5]

    # Rate of change
    df['nilai_roc'] = (df[nilai_cols].iloc[:, -1] - df[nilai_cols].iloc[:, -2]) / (df[nilai_cols].iloc[:, -2] + 1e-6)
    df['aktivitas_roc'] = (df[aktivitas_cols].iloc[:, -1] - df[aktivitas_cols].iloc[:, -2]) / (df[aktivitas_cols].iloc[:, -2] + 1e-6)

    # Composite scores
    df['academic_score'] = (
        df['nilai_mean'] * 0.4 +
        df['skor_tryout'] * 0.3 +
        df['total_skor'] * 0.2 +
        df['rasio_tugas'] * 0.1
    )

    df['behavioral_score'] = (
        df['aktivitas_mean'] * 0.4 +
        df['indeks_kehadiran'] * 0.3 +
        df['skor_kedisiplinan'] * 0.2 +
        df['skor_minat_belajar'] * 0.1
    )

    df['overall_score'] = df['academic_score'] * 0.6 + df['behavioral_score'] * 0.4

    # Interaction between academic and behavioral
    df['academic_behavioral_interaction'] = df['academic_score'] * df['behavioral_score']

    # Stability metrics
    df['nilai_stability'] = 1 - (df['nilai_std'] / (df['nilai_max'] - df['nilai_min'] + 1e-6))
    df['aktivitas_stability'] = 1 - (df['aktivitas_std'] / (df['aktivitas_max'] - df['aktivitas_min'] + 1e-6))

    # Recent performance focus
    df['recent_nilai_avg'] = df[nilai_cols].iloc[:, -4:].mean(axis=1)
    df['recent_aktivitas_avg'] = df[aktivitas_cols].iloc[:, -4:].mean(axis=1)

    # Early vs late performance comparison
    df['early_nilai_avg'] = df[nilai_cols].iloc[:, :4].mean(axis=1)
    df['late_vs_early_nilai'] = df['recent_nilai_avg'] - df['early_nilai_avg']

    # Fill any NaN values
    df = df.fillna(0)

    return df

# Apply feature engineering
train_fe = create_features(train)
test_fe = create_features(test)

print(f"Features setelah engineering: {train_fe.shape}")

Features setelah engineering: (3200, 104)


## 2. Split Data dan Handle Class Imbalance

In [4]:
# Tentukan fitur yang akan digunakan
drop_cols = ['id', 'target']
feature_cols = [col for col in train_fe.columns if col not in drop_cols]

X = train_fe[feature_cols]
y = train_fe['target']
X_test = test_fe[feature_cols]

# Split data dengan stratifikasi
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")

# Handle Class Imbalance dengan SMOTE
print("\nApplying SMOTE to handle class imbalance...")
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f"Original training set shape: {X_train.shape}")
print(f"After SMOTE training set shape: {X_train_smote.shape}")
print(f"\nOriginal target distribution:")
print(pd.Series(y_train).value_counts())
print(f"\nAfter SMOTE target distribution:")
print(pd.Series(y_train_smote).value_counts())

Training set: (2560, 102)
Validation set: (640, 102)

Applying SMOTE to handle class imbalance...
Original training set shape: (2560, 102)
After SMOTE training set shape: (2600, 102)

Original target distribution:
target
0    650
3    646
1    637
2    627
Name: count, dtype: int64

After SMOTE target distribution:
target
3    650
0    650
2    650
1    650
Name: count, dtype: int64


## 3. Training Multiple Models Alternatif

In [5]:
print("=== Training Multiple Models ===\n")

# 1. Random Forest
rf = RandomForestClassifier(n_estimators=300, max_depth=15, random_state=42, n_jobs=-1)
rf.fit(X_train_smote, y_train_smote)
rf_pred = rf.predict(X_val)
rf_acc = accuracy_score(y_val, rf_pred)
print(f"Random Forest Validation Accuracy: {rf_acc:.4f}")

# 2. LightGBM
lgbm = LGBMClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.1,
    random_state=42,
    verbose=-1
)
lgbm.fit(X_train_smote, y_train_smote)
lgbm_pred = lgbm.predict(X_val)
lgbm_acc = accuracy_score(y_val, lgbm_pred)
print(f"LightGBM Validation Accuracy: {lgbm_acc:.4f}")

# 3. CatBoost
cat = CatBoostClassifier(
    iterations=300,
    depth=8,
    learning_rate=0.1,
    random_state=42,
    verbose=False
)
cat.fit(X_train_smote, y_train_smote)
cat_pred = cat.predict(X_val)
cat_acc = accuracy_score(y_val, cat_pred)
print(f"CatBoost Validation Accuracy: {cat_acc:.4f}")

# 4. Gradient Boosting
gb = GradientBoostingClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.1,
    random_state=42
)
gb.fit(X_train_smote, y_train_smote)
gb_pred = gb.predict(X_val)
gb_acc = accuracy_score(y_val, gb_pred)
print(f"Gradient Boosting Validation Accuracy: {gb_acc:.4f}")

print(f"\nBest single model so far: {max(rf_acc, lgbm_acc, cat_acc, gb_acc):.4f}")

=== Training Multiple Models ===

Random Forest Validation Accuracy: 0.4703
LightGBM Validation Accuracy: 0.5141
CatBoost Validation Accuracy: 0.5437
Gradient Boosting Validation Accuracy: 0.4750

Best single model so far: 0.5437


## 4. Ensemble Methods

In [6]:
print("=== Ensemble Methods ===\n")

# 1. Voting Ensemble (Soft Voting)
voting_clf = VotingClassifier(
    estimators=[
        ('xgb', XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42, use_label_encoder=False, eval_metric='mlogloss')),
        ('rf', RandomForestClassifier(n_estimators=300, max_depth=15, random_state=42, n_jobs=-1)),
        ('lgbm', LGBMClassifier(n_estimators=300, max_depth=8, learning_rate=0.1, random_state=42, verbose=-1)),
        ('cat', CatBoostClassifier(iterations=300, depth=8, learning_rate=0.1, random_state=42, verbose=False))
    ],
    voting='soft',
    n_jobs=-1
)

print("Training Voting Classifier...")
voting_clf.fit(X_train_smote, y_train_smote)
voting_pred = voting_clf.predict(X_val)
voting_acc = accuracy_score(y_val, voting_pred)
print(f"Voting Classifier Validation Accuracy: {voting_acc:.4f}")

# 2. Stacking Classifier
stacking_clf = StackingClassifier(
    estimators=[
        ('xgb', XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42, use_label_encoder=False, eval_metric='mlogloss')),
        ('rf', RandomForestClassifier(n_estimators=300, max_depth=15, random_state=42, n_jobs=-1)),
        ('lgbm', LGBMClassifier(n_estimators=300, max_depth=8, learning_rate=0.1, random_state=42, verbose=-1))
    ],
    final_estimator=LogisticRegression(max_iter=1000, random_state=42),
    cv=3,
    n_jobs=-1
)

print("\nTraining Stacking Classifier...")
stacking_clf.fit(X_train_smote, y_train_smote)
stacking_pred = stacking_clf.predict(X_val)
stacking_acc = accuracy_score(y_val, stacking_pred)
print(f"Stacking Classifier Validation Accuracy: {stacking_acc:.4f}")

print(f"\nBest ensemble method: {max(voting_acc, stacking_acc):.4f}")

=== Ensemble Methods ===

Training Voting Classifier...
Voting Classifier Validation Accuracy: 0.5266

Training Stacking Classifier...
Stacking Classifier Validation Accuracy: 0.5250

Best ensemble method: 0.5266


## 5. CatBoost-Specific Hyperparameter Tuning

In [7]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint
from catboost import CatBoostClassifier
import time

print("=== CatBoost-Specific Hyperparameter Tuning ===\n")

# Pastikan X_train_smote dan y_train_smote sudah terdefinisi
# Jika belum, run cell 5 terlebih dahulu

try:
    X_train_smote
    y_train_smote
    print("X_train_smote dan y_train_smote sudah terdefinisi")
except NameError:
    print("ERROR: X_train_smote dan y_train_smote belum terdefinisi!")
    print("Silakan run cell 5 (Split Data dan Handle Class Imbalance) terlebih dahulu.")
    print("Atau run semua cell dari awal (Cell 1, 3, 5) secara berurutan.")
    raise

# === STRATEGI 1: Coarse Search (Pencarian Kasar) ===
print("Stage 1: Coarse Search (Mencari area terbaik)...")
start_time = time.time()

# Parameter distribution yang lebih luas untuk coarse search
param_dist_cat_coarse = {
    'iterations': randint(100, 500),      # Range lebih luas
    'depth': randint(3, 10),
    'learning_rate': uniform(0.01, 0.3),
    'l2_leaf_reg': uniform(1, 10),
    'border_count': randint(32, 200),
    'bagging_temperature': uniform(0, 1),
    'random_strength': uniform(0, 1),
    'one_hot_max_size': randint(2, 10)
}

cat_search_coarse = RandomizedSearchCV(
    CatBoostClassifier(random_state=42, verbose=False),
    param_distributions=param_dist_cat_coarse,
    n_iter=20,          # Dikurangi dari 50
    cv=3,               # Dikurangi dari 5
    scoring='accuracy',
    random_state=42,
    n_jobs=-1,          # Gunakan semua CPU core
    verbose=1
)

print("Melakukan coarse search...")
cat_search_coarse.fit(X_train_smote, y_train_smote)

coarse_time = time.time() - start_time
print(f"\nCoarse search selesai dalam {coarse_time:.2f} detik")
print(f"Best coarse parameters: {cat_search_coarse.best_params_}")
print(f"Best coarse CV accuracy: {cat_search_coarse.best_score_:.4f}")

# === STRATEGI 2: Fine Search (Pencarian Halus) ===
print("\n" + "="*50)
print("Stage 2: Fine Search (Memperhalus parameter)...")
start_time = time.time()

# Ambil best parameters dari coarse search
best_coarse = cat_search_coarse.best_params_

# Buat parameter distribution yang lebih spesifik di sekitar best parameters
param_dist_cat_fine = {
    'iterations': randint(
        max(50, best_coarse['iterations'] - 100),
        best_coarse['iterations'] + 100
    ),
    'depth': randint(
        max(2, best_coarse['depth'] - 2),
        min(12, best_coarse['depth'] + 2)
    ),
    'learning_rate': uniform(
        max(0.005, best_coarse['learning_rate'] - 0.05),
        min(0.1, best_coarse['learning_rate'] + 0.1)
    ),
    'l2_leaf_reg': uniform(
        max(0.5, best_coarse['l2_leaf_reg'] - 2),
        min(5, best_coarse['l2_leaf_reg'] + 2)
    ),
    'border_count': randint(
        max(16, best_coarse['border_count'] - 50),
        min(255, best_coarse['border_count'] + 50)
    ),
    'bagging_temperature': uniform(
        max(0, best_coarse['bagging_temperature'] - 0.2),
        min(0.4, best_coarse['bagging_temperature'] + 0.2)
    ),
    'random_strength': uniform(
        max(0, best_coarse['random_strength'] - 0.2),
        min(0.4, best_coarse['random_strength'] + 0.2)
    ),
    'one_hot_max_size': randint(
        max(1, best_coarse['one_hot_max_size'] - 3),
        min(15, best_coarse['one_hot_max_size'] + 3)
    )
}

cat_search_fine = RandomizedSearchCV(
    CatBoostClassifier(random_state=42, verbose=False),
    param_distributions=param_dist_cat_fine,
    n_iter=15,          # Lebih sedikit untuk fine-tuning
    cv=5,               # Kembali ke 5 folds untuk hasil lebih akurat
    scoring='accuracy',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

print("Melakukan fine search...")
cat_search_fine.fit(X_train_smote, y_train_smote)

fine_time = time.time() - start_time
print(f"\nFine search selesai dalam {fine_time:.2f} detik")
print(f"Best fine parameters: {cat_search_fine.best_params_}")
print(f"Best fine CV accuracy: {cat_search_fine.best_score_:.4f}")

# === EVALUASI FINAL ===
print("\n" + "="*50)
print("=== Evaluasi Final ===")

# Pilih model terbaik dari fine search
best_cat_model = cat_search_fine.best_estimator_

# Evaluasi pada validation set
cat_tuned_pred = best_cat_model.predict(X_val)
cat_tuned_acc = accuracy_score(y_val, cat_tuned_pred)

print(f"Validation Accuracy dengan tuned CatBoost: {cat_tuned_acc:.4f}")

# === PERBANDINGAN WAKTU ===
total_time = coarse_time + fine_time
print(f"\nTotal waktu tuning: {total_time:.2f} detik ({total_time/60:.2f} menit)")

# Estimasi waktu jika menggunakan 50 candidates × 5 folds
estimated_old_time = (total_time / (20*3 + 15*5)) * (50*5)
print(f"Estimasi waktu dengan metode lama (50 candidates × 5 folds): {estimated_old_time:.2f} detik ({estimated_old_time/60:.2f} menit)")
print(f"Penghematan waktu: {(1 - total_time/estimated_old_time)*100:.1f}%")

=== CatBoost-Specific Hyperparameter Tuning ===

X_train_smote dan y_train_smote sudah terdefinisi
Stage 1: Coarse Search (Mencari area terbaik)...
Melakukan coarse search...
Fitting 3 folds for each of 20 candidates, totalling 60 fits

Coarse search selesai dalam 1227.88 detik
Best coarse parameters: {'bagging_temperature': np.float64(0.07404465173409036), 'border_count': 72, 'depth': 6, 'iterations': 234, 'l2_leaf_reg': np.float64(9.500385777897993), 'learning_rate': np.float64(0.14483520224146101), 'one_hot_max_size': 2, 'random_strength': np.float64(0.06355835028602363)}
Best coarse CV accuracy: 0.5435

Stage 2: Fine Search (Memperhalus parameter)...
Melakukan fine search...
Fitting 5 folds for each of 15 candidates, totalling 75 fits

Fine search selesai dalam 438.40 detik
Best fine parameters: {'bagging_temperature': np.float64(0.1664944173521692), 'border_count': 42, 'depth': 4, 'iterations': 300, 'l2_leaf_reg': np.float64(7.566710583697326), 'learning_rate': np.float64(0.189055

## 6. Robust Cross-Validation dengan Repeated Stratified K-Fold

In [8]:
print("=== Robust Cross-Validation dengan Repeated Stratified K-Fold ===\n")

from sklearn.model_selection import RepeatedStratifiedKFold
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import cross_val_score

# Gunakan Repeated Stratified K-Fold untuk mengurangi variance
rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)

# Gunakan CatBoost sederhana untuk cross-validation (tanpa parameter yang menyebabkan cloning error)
cat_for_cv = CatBoostClassifier(
    iterations=300,
    depth=8,
    learning_rate=0.1,
    random_state=42,
    verbose=False
)

print("Evaluasi CatBoost dengan robust CV...")
cat_cv_scores = cross_val_score(
    cat_for_cv,
    X_train_smote,
    y_train_smote,
    cv=rskf,
    scoring='accuracy',
    n_jobs=-1
)

print(f"CatBoost Repeated Stratified K-Fold CV:")
print(f"Mean CV: {cat_cv_scores.mean():.4f} (+/- {cat_cv_scores.std():.4f})")
print(f"Min CV: {cat_cv_scores.min():.4f}")
print(f"Max CV: {cat_cv_scores.max():.4f}")

# Evaluasi LightGBM dengan robust CV (model terbaik kedua)
lgbm_for_cv = LGBMClassifier(n_estimators=300, max_depth=8, learning_rate=0.1, random_state=42, verbose=-1)
lgbm_cv_scores = cross_val_score(
    lgbm_for_cv,
    X_train_smote,
    y_train_smote,
    cv=rskf,
    scoring='accuracy',
    n_jobs=-1
)

print(f"\nLightGBM Repeated Stratified K-Fold CV:")
print(f"Mean CV: {lgbm_cv_scores.mean():.4f} (+/- {lgbm_cv_scores.std():.4f})")

print(f"\nCatatan: Cross-validation menggunakan model CatBoost sederhana untuk menghindari cloning error.")
print(f"Model CatBoost yang di-tune (best_cat_model) akan digunakan untuk final submission.")

=== Robust Cross-Validation dengan Repeated Stratified K-Fold ===

Evaluasi CatBoost dengan robust CV...
CatBoost Repeated Stratified K-Fold CV:
Mean CV: 0.5332 (+/- 0.0164)
Min CV: 0.4981
Max CV: 0.5596

LightGBM Repeated Stratified K-Fold CV:
Mean CV: 0.5106 (+/- 0.0142)

Catatan: Cross-validation menggunakan model CatBoost sederhana untuk menghindari cloning error.
Model CatBoost yang di-tune (best_cat_model) akan digunakan untuk final submission.


## 7. Weighted Ensemble dengan Top 2 Models (CatBoost + LightGBM)

In [9]:
print("=== Weighted Ensemble dengan Top 2 Models ===\n")

# Gunakan CatBoost sederhana untuk ensemble (untuk menghindari cloning error)
cat_simple = CatBoostClassifier(
    iterations=300,
    depth=8,
    learning_rate=0.1,
    random_state=42,
    verbose=False
)

# Ensemble CatBoost + LightGBM dengan weighted voting
voting_top2 = VotingClassifier(
    estimators=[
        ('cat', cat_simple),  # CatBoost sederhana
        ('lgbm', LGBMClassifier(n_estimators=300, max_depth=8, learning_rate=0.1, random_state=42, verbose=-1))
    ],
    voting='soft',
    weights=[0.6, 0.4],  # Beri lebih banyak weight ke CatBoost
    n_jobs=-1
)

print("Training Weighted Voting Classifier (CatBoost + LightGBM)...")
voting_top2.fit(X_train_smote, y_train_smote)
voting_top2_pred = voting_top2.predict(X_val)
voting_top2_acc = accuracy_score(y_val, voting_top2_pred)
print(f"Weighted Voting Validation Accuracy: {voting_top2_acc:.4f}")

# Coba juga dengan weights berbeda
for w1, w2 in [(0.7, 0.3), (0.5, 0.5), (0.8, 0.2)]:
    voting_test = VotingClassifier(
        estimators=[
            ('cat', cat_simple),
            ('lgbm', LGBMClassifier(n_estimators=300, max_depth=8, learning_rate=0.1, random_state=42, verbose=-1))
        ],
        voting='soft',
        weights=[w1, w2],
        n_jobs=-1
    )
    voting_test.fit(X_train_smote, y_train_smote)
    voting_test_pred = voting_test.predict(X_val)
    voting_test_acc = accuracy_score(y_val, voting_test_pred)
    print(f"Weighted Voting (CatBoost={w1}, LightGBM={w2}): {voting_test_acc:.4f}")

print(f"\nCatatan: Ensemble menggunakan CatBoost sederhana untuk menghindari cloning error.")
print(f"Model CatBoost yang di-tune (best_cat_model) akan digunakan untuk final submission.")

=== Weighted Ensemble dengan Top 2 Models ===

Training Weighted Voting Classifier (CatBoost + LightGBM)...
Weighted Voting Validation Accuracy: 0.5297
Weighted Voting (CatBoost=0.7, LightGBM=0.3): 0.5344
Weighted Voting (CatBoost=0.5, LightGBM=0.5): 0.5219
Weighted Voting (CatBoost=0.8, LightGBM=0.2): 0.5422

Catatan: Ensemble menggunakan CatBoost sederhana untuk menghindari cloning error.
Model CatBoost yang di-tune (best_cat_model) akan digunakan untuk final submission.


In [10]:
print("=== Calibration untuk Post-Processing ===\n")

from sklearn.calibration import CalibratedClassifierCV

# Gunakan CatBoost sederhana untuk calibration (untuk menghindari cloning error)
cat_simple = CatBoostClassifier(
    iterations=300,
    depth=8,
    learning_rate=0.1,
    random_state=42,
    verbose=False
)

# Calibrate CatBoost probabilities untuk improve prediction
calibrated_cat = CalibratedClassifierCV(
    cat_simple,
    method='isotonic',
    cv=5
)

print("Training Calibrated CatBoost...")
calibrated_cat.fit(X_train_smote, y_train_smote)
calibrated_cat_pred = calibrated_cat.predict(X_val)
calibrated_cat_acc = accuracy_score(y_val, calibrated_cat_pred)
print(f"Calibrated CatBoost Validation Accuracy: {calibrated_cat_acc:.4f}")

# Coba juga dengan sigmoid calibration
calibrated_cat_sigmoid = CalibratedClassifierCV(
    cat_simple,
    method='sigmoid',
    cv=5
)

print("\nTraining Calibrated CatBoost (Sigmoid)...")
calibrated_cat_sigmoid.fit(X_train_smote, y_train_smote)
calibrated_cat_sigmoid_pred = calibrated_cat_sigmoid.predict(X_val)
calibrated_cat_sigmoid_acc = accuracy_score(y_val, calibrated_cat_sigmoid_pred)
print(f"Calibrated CatBoost (Sigmoid) Validation Accuracy: {calibrated_cat_sigmoid_acc:.4f}")

print(f"\nCatatan: Calibration menggunakan CatBoost sederhana untuk menghindari cloning error.")

=== Calibration untuk Post-Processing ===

Training Calibrated CatBoost...
Calibrated CatBoost Validation Accuracy: 0.5453

Training Calibrated CatBoost (Sigmoid)...
Calibrated CatBoost (Sigmoid) Validation Accuracy: 0.5391

Catatan: Calibration menggunakan CatBoost sederhana untuk menghindari cloning error.


## Summary

Notebook ini telah menerapkan strategi improve dari baseline 49% ke target 70-80%:

### Improvements yang Diterapkan:
1. **Advanced Feature Engineering** - rolling statistics, trend analysis, polynomial features
2. **Domain-Specific Features** - consistency metrics, improvement rate, academic potential score, learning momentum
3. **SMOTE** - untuk handle class imbalance
4. **Multiple Model Alternatives** - Random Forest, LightGBM, CatBoost, Gradient Boosting
5. **CatBoost-Specific Hyperparameter Tuning** - RandomizedSearchCV dengan 50 iterasi
6. **Robust Cross-Validation** - Repeated Stratified K-Fold (5 splits, 3 repeats)
7. **Weighted Ensemble** - Top 2 models (CatBoost + LightGBM) dengan weighted voting
8. **Calibration** - Isotonic dan sigmoid calibration untuk post-processing

### Hasil Eksperimen (sebelum improvement):
- Baseline: 49.38%
- CatBoost single: 53.75%
- Voting Classifier: 51.25%

### Target Improvement:
- **Target**: 70-80% (dengan advanced feature engineering + ensemble + optimal strategy)

### Submission Files:
- `outputs/submission_final.csv` - CatBoost tuned (recommended)
- `outputs/submission_all_strategies.csv` - Semua model predictions untuk comparison

### Catatan Penting:
- Feature selection TIDAK digunakan karena menurunkan akurasi
- Fokus pada CatBoost karena sudah terbukti terbaik
- Ensemble hanya efektif dengan 2 model terbaik
- Robust CV penting untuk menghindari overfitting

In [11]:
print("=== Generate Final Submission dengan Best Blending ===\n")

# Apply SMOTE pada full training data
smote_full = SMOTE(random_state=42)
X_full_smote, y_full_smote = smote_full.fit_resample(X, y)

# Train CatBoost dengan best tuned parameters pada full dataset
print("Training CatBoost with best tuned parameters on full dataset...")
cat_final = CatBoostClassifier(
    iterations=436,
    depth=5,
    learning_rate=0.16015757296260286,
    l2_leaf_reg=11.734628978618437,
    border_count=228,
    bagging_temperature=0.06423578631931819,
    random_strength=0.33169390030136725,
    one_hot_max_size=2,
    random_state=42,
    verbose=False
)
cat_final.fit(X_full_smote, y_full_smote)

# Train LightGBM pada full dataset
print("Training LightGBM on full dataset...")
lgbm_final = LGBMClassifier(n_estimators=300, max_depth=8, learning_rate=0.1, random_state=42, verbose=-1)
lgbm_final.fit(X_full_smote, y_full_smote)

# Get probabilities for test set
cat_proba_test = cat_final.predict_proba(X_test)
lgbm_proba_test = lgbm_final.predict_proba(X_test)

# Use best weights dari manual blending (0.75, 0.25)
best_w_cat, best_w_lgbm = 0.75, 0.25

# Blending
blended_proba_test = best_w_cat * cat_proba_test + best_w_lgbm * lgbm_proba_test
test_predictions = np.argmax(blended_proba_test, axis=1)

# Buat submission file
submission = pd.DataFrame({
    'id': test['id'],
    'target': test_predictions
})

submission.to_csv('submission_final.csv', index=False)
print("Submission file saved as 'submission_final.csv'")
print(f"\nSubmission shape: {submission.shape}")
print(submission.head())

# Simpan juga individual predictions untuk comparison
submission['target_catboost'] = np.argmax(cat_proba_test, axis=1)
submission['target_lgbm'] = np.argmax(lgbm_proba_test, axis=1)
submission.to_csv('submission_all_strategies.csv', index=False)
print("\nAll strategy predictions saved as 'submission_all_strategies.csv'")

print(f"\n=== Summary ===")
print(f"Submission strategy: Blending (CatBoost={best_w_cat}, LightGBM={best_w_lgbm})")
print(f"Files generated:")
print("1. submission_final.csv - Best blending (recommended)")
print("2. submission_all_strategies.csv - Individual predictions for comparison")

=== Generate Final Submission dengan Best Blending ===

Training CatBoost with best tuned parameters on full dataset...
Training LightGBM on full dataset...
Submission file saved as 'submission_final.csv'

Submission shape: (800, 2)
   id  target
0   3       0
1  12       2
2  14       3
3  18       3
4  28       0

All strategy predictions saved as 'submission_all_strategies.csv'

=== Summary ===
Submission strategy: Blending (CatBoost=0.75, LightGBM=0.25)
Files generated:
1. submission_final.csv - Best blending (recommended)
2. submission_all_strategies.csv - Individual predictions for comparison


In [12]:
print("=== Advanced Feature Selection ===\n")

from sklearn.feature_selection import mutual_info_classif, SelectFromModel
import matplotlib.pyplot as plt

# 1. Mutual Information
print("Calculating Mutual Information...")
mi_scores = mutual_info_classif(X_train_smote, y_train_smote)
mi_df = pd.DataFrame({'feature': X_train_smote.columns, 'mi_score': mi_scores})
mi_df = mi_df.sort_values('mi_score', ascending=False)
print(f"Top 20 features by Mutual Information:")
print(mi_df.head(20))

# Select top features based on MI
top_n = 60
top_features = mi_df.head(top_n)['feature'].tolist()
print(f"\nSelected top {top_n} features based on Mutual Information")

X_train_selected = X_train_smote[top_features]
X_val_selected = X_val[top_features]
X_test_selected = X_test[top_features]

print(f"Features after selection: {X_train_selected.shape}")

# Train CatBoost on selected features
print("\nTraining CatBoost on selected features...")
cat_selected = CatBoostClassifier(
    iterations=436, depth=5, learning_rate=0.16,
    l2_leaf_reg=11.73, border_count=228, random_state=42, verbose=False
)
cat_selected.fit(X_train_selected, y_train_smote)
cat_selected_pred = cat_selected.predict(X_val_selected)
cat_selected_acc = accuracy_score(y_val, cat_selected_pred)
print(f"CatBoost with Feature Selection Validation Accuracy: {cat_selected_acc:.4f}")

# Compare with original
print(f"Original CatBoost Accuracy: {cat_acc:.4f}")
print(f"Improvement: {(cat_selected_acc - cat_acc)*100:.2f}%")

=== Advanced Feature Selection ===

Calculating Mutual Information...
Top 20 features by Mutual Information:
                   feature  mi_score
71     rasio_tugas_squared  0.102490
61             rasio_tugas  0.095676
45             nilai_range  0.089942
76         task_efficiency  0.088492
101    late_vs_early_nilai  0.085634
100        early_nilai_avg  0.080841
53               nilai_iqr  0.075521
81     nilai_percentile_10  0.071925
43               nilai_max  0.071896
50        nilai_volatility  0.070064
42               nilai_std  0.069979
52              nilai_25th  0.068590
44               nilai_min  0.068333
80     nilai_percentile_90  0.068078
63   motivasi_kedisiplinan  0.067130
7          nilai_minggu_08  0.059858
98        recent_nilai_avg  0.055977
2          nilai_minggu_03  0.053599
79       learning_momentum  0.041877
0          nilai_minggu_01  0.041552

Selected top 60 features based on Mutual Information
Features after selection: (2600, 60)

Training CatBoost on s

In [13]:
print("=== Neural Network Approach with MLPClassifier ===\n")

from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

# Scale features for neural network
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_smote)
X_val_scaled = scaler.transform(X_val)

# Try different neural network architectures
nn_configs = [
    {'hidden_layer_sizes': (100, 50), 'alpha': 0.001, 'learning_rate': 'adaptive'},
    {'hidden_layer_sizes': (150, 75, 30), 'alpha': 0.0001, 'learning_rate': 'adaptive'},
    {'hidden_layer_sizes': (200, 100, 50), 'alpha': 0.0001, 'learning_rate': 'adaptive'},
]

best_nn_acc = 0
best_nn_model = None

for i, config in enumerate(nn_configs):
    print(f"Training Neural Network {i+1} with config: {config}")
    nn = MLPClassifier(
        hidden_layer_sizes=config['hidden_layer_sizes'],
        alpha=config['alpha'],
        learning_rate=config['learning_rate'],
        max_iter=500,
        random_state=42,
        early_stopping=True,
        validation_fraction=0.1
    )
    nn.fit(X_train_scaled, y_train_smote)
    nn_pred = nn.predict(X_val_scaled)
    nn_acc = accuracy_score(y_val, nn_pred)
    print(f"Neural Network {i+1} Validation Accuracy: {nn_acc:.4f}")

    if nn_acc > best_nn_acc:
        best_nn_acc = nn_acc
        best_nn_model = nn

print(f"\nBest Neural Network Accuracy: {best_nn_acc:.4f}")

=== Neural Network Approach with MLPClassifier ===

Training Neural Network 1 with config: {'hidden_layer_sizes': (100, 50), 'alpha': 0.001, 'learning_rate': 'adaptive'}
Neural Network 1 Validation Accuracy: 0.5156
Training Neural Network 2 with config: {'hidden_layer_sizes': (150, 75, 30), 'alpha': 0.0001, 'learning_rate': 'adaptive'}
Neural Network 2 Validation Accuracy: 0.5234
Training Neural Network 3 with config: {'hidden_layer_sizes': (200, 100, 50), 'alpha': 0.0001, 'learning_rate': 'adaptive'}
Neural Network 3 Validation Accuracy: 0.5141

Best Neural Network Accuracy: 0.5234


In [14]:
print("=== Feature Importance Analysis dan Noisy Feature Removal ===\n")

# Train CatBoost to get feature importance
cat_for_importance = CatBoostClassifier(
    iterations=300, depth=8, learning_rate=0.1,
    random_state=42, verbose=False
)
cat_for_importance.fit(X_train_smote, y_train_smote)

# Get feature importance
feature_importance = cat_for_importance.get_feature_importance()
feature_names = X_train_smote.columns
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': feature_importance
}).sort_values('importance', ascending=False)

print("Top 30 most important features:")
print(importance_df.head(30))

print("\nBottom 10 least important features:")
print(importance_df.tail(10))

# Remove features with very low importance
importance_threshold = 0.5  # Features with importance < 0.5 will be removed
important_features = importance_df[importance_df['importance'] >= importance_threshold]['feature'].tolist()
print(f"\nFeatures with importance >= {importance_threshold}: {len(important_features)}")
print(f"Features to remove: {len(feature_names) - len(important_features)}")

# Train model with only important features
X_train_important = X_train_smote[important_features]
X_val_important = X_val[important_features]
X_test_important = X_test[important_features]

print(f"\nTraining CatBoost with important features only...")
cat_important = CatBoostClassifier(
    iterations=300, depth=8, learning_rate=0.1,
    random_state=42, verbose=False
)
cat_important.fit(X_train_important, y_train_smote)
cat_important_pred = cat_important.predict(X_val_important)
cat_important_acc = accuracy_score(y_val, cat_important_pred)
print(f"CatBoost with important features: {cat_important_acc:.4f}")
print(f"Original CatBoost accuracy: {cat_acc:.4f}")
print(f"Improvement: {(cat_important_acc - cat_acc)*100:.2f}%")

# Try different thresholds
print("\n=== Testing different importance thresholds ===")
thresholds = [0.1, 0.5, 1.0, 2.0, 3.0]
threshold_results = {}

for threshold in thresholds:
    features = importance_df[importance_df['importance'] >= threshold]['feature'].tolist()
    if len(features) < 10:  # Skip if too few features
        continue

    X_train_thresh = X_train_smote[features]
    X_val_thresh = X_val[features]

    cat_thresh = CatBoostClassifier(iterations=300, depth=8, learning_rate=0.1, random_state=42, verbose=False)
    cat_thresh.fit(X_train_thresh, y_train_smote)
    pred_thresh = cat_thresh.predict(X_val_thresh)
    acc_thresh = accuracy_score(y_val, pred_thresh)
    threshold_results[threshold] = (acc_thresh, len(features))
    print(f"Threshold {threshold}: {acc_thresh:.4f} ({len(features)} features)")

# Find best threshold
best_threshold = max(threshold_results, key=lambda x: threshold_results[x][0])
best_acc, best_n_features = threshold_results[best_threshold]
print(f"\nBest threshold: {best_threshold} with accuracy {best_acc:.4f} ({best_n_features} features)")

# Use best features for final training
best_features = importance_df[importance_df['importance'] >= best_threshold]['feature'].tolist()
print(f"\nSelected {len(best_features)} features with threshold {best_threshold}")

=== Feature Importance Analysis dan Noisy Feature Removal ===

Top 30 most important features:
                     feature  importance
63     motivasi_kedisiplinan    7.999229
71       rasio_tugas_squared    3.626122
61               rasio_tugas    3.580450
76           task_efficiency    3.310620
18         aktivitas_hari_05    2.183851
24         aktivitas_hari_11    2.094477
42                 nilai_std    1.994744
21         aktivitas_hari_08    1.901547
26         aktivitas_hari_13    1.831553
60   aktivitas_rolling_std_3    1.702327
17         aktivitas_hari_04    1.638466
20         aktivitas_hari_07    1.631350
39            jumlah_saudara    1.615337
89      aktivitas_momentum_5    1.592533
15         aktivitas_hari_02    1.558551
53                 nilai_iqr    1.377111
44                 nilai_min    1.372123
38             skor_literasi    1.302707
19         aktivitas_hari_06    1.261519
25         aktivitas_hari_12    1.246808
33              urutan_ujian    1.237396
59 

In [15]:
print("=== Advanced Stacking dengan Meta-Learners ===\n")

from sklearn.ensemble import StackingClassifier, RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

# Define base learners with diverse algorithms
base_learners = [
    ('catboost', CatBoostClassifier(iterations=300, depth=8, learning_rate=0.1, random_state=42, verbose=False)),
    ('lgbm', LGBMClassifier(n_estimators=300, max_depth=8, learning_rate=0.1, random_state=42, verbose=-1)),
    ('xgboost', XGBClassifier(n_estimators=300, max_depth=8, learning_rate=0.1, random_state=42, use_label_encoder=False, eval_metric='mlogloss')),
    ('rf', RandomForestClassifier(n_estimators=300, max_depth=12, random_state=42, n_jobs=-1)),
    ('gb', GradientBoostingClassifier(n_estimators=300, max_depth=8, learning_rate=0.1, random_state=42)),
]

# Try different meta-learners
meta_learners = {
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
    'RidgeClassifier': RidgeClassifier(random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42, n_jobs=-1),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42),
    'SVC': SVC(probability=True, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
}

stacking_results = {}

print("Training Stacking Classifiers with different meta-learners...\n")
for meta_name, meta_learner in meta_learners.items():
    try:
        print(f"Training Stacking with {meta_name} as meta-learner...")
        stacking_clf = StackingClassifier(
            estimators=base_learners,
            final_estimator=meta_learner,
            cv=3,
            n_jobs=-1,
            stack_method='predict_proba'
        )

        stacking_clf.fit(X_train_smote, y_train_smote)
        stacking_pred = stacking_clf.predict(X_val)
        stacking_acc = accuracy_score(y_val, stacking_pred)
        stacking_results[meta_name] = stacking_acc
        print(f"Stacking with {meta_name}: {stacking_acc:.4f}\n")

    except Exception as e:
        print(f"Stacking with {meta_name} failed: {e}\n")

# Find best meta-learner
best_meta = max(stacking_results, key=stacking_results.get)
print(f"Best meta-learner: {best_meta} with accuracy {stacking_results[best_meta]:.4f}")

# Train final stacking with best meta-learner
print(f"\nTraining final Stacking Classifier with {best_meta}...")
final_stacking = StackingClassifier(
    estimators=base_learners,
    final_estimator=meta_learners[best_meta],
    cv=5,
    n_jobs=-1,
    stack_method='predict_proba'
)
final_stacking.fit(X_train_smote, y_train_smote)
final_stacking_pred = final_stacking.predict(X_val)
final_stacking_acc = accuracy_score(y_val, final_stacking_pred)
print(f"Final Stacking Validation Accuracy: {final_stacking_acc:.4f}")

# Get meta-features for analysis
print("\nAnalyzing meta-features...")
meta_features_train = final_stacking.transform(X_train_smote)
meta_features_val = final_stacking.transform(X_val)
print(f"Meta-features shape: {meta_features_train.shape}")

# Train simple model on meta-features
meta_lr = LogisticRegression(max_iter=1000, random_state=42)
meta_lr.fit(meta_features_train, y_train_smote)
meta_pred = meta_lr.predict(meta_features_val)
meta_acc = accuracy_score(y_val, meta_pred)
print(f"Logistic Regression on meta-features: {meta_acc:.4f}")

=== Advanced Stacking dengan Meta-Learners ===

Training Stacking Classifiers with different meta-learners...

Training Stacking with LogisticRegression as meta-learner...
Stacking with LogisticRegression: 0.5625

Training Stacking with RidgeClassifier as meta-learner...
Stacking with RidgeClassifier: 0.5516

Training Stacking with RandomForest as meta-learner...
Stacking with RandomForest: 0.5656

Training Stacking with GradientBoosting as meta-learner...
Stacking with GradientBoosting: 0.5656

Training Stacking with SVC as meta-learner...
Stacking with SVC: 0.5641

Training Stacking with KNN as meta-learner...
Stacking with KNN: 0.4703

Best meta-learner: RandomForest with accuracy 0.5656

Training final Stacking Classifier with RandomForest...
Final Stacking Validation Accuracy: 0.5547

Analyzing meta-features...
Meta-features shape: (2600, 20)
Logistic Regression on meta-features: 0.5062


In [16]:
print("=== Data Augmentation dengan SMOTE Variants dan ADASYN ===\n")

from imblearn.over_sampling import SMOTE, ADASYN, BorderlineSMOTE, SVMSMOTE, SMOTENC
from imblearn.combine import SMOTETomek, SMOTEENN

# Try different oversampling techniques
oversamplers = {
    'SMOTE': SMOTE(random_state=42),
    'ADASYN': ADASYN(random_state=42),
    'BorderlineSMOTE': BorderlineSMOTE(random_state=42),
    'SVMSMOTE': SVMSMOTE(random_state=42),
    'SMOTETomek': SMOTETomek(random_state=42),
    'SMOTEENN': SMOTEENN(random_state=42)
}

results = {}

print("Testing different oversampling techniques...\n")
for name, sampler in oversamplers.items():
    try:
        print(f"Applying {name}...")
        X_resampled, y_resampled = sampler.fit_resample(X_train, y_train)

        # Train CatBoost on resampled data
        cat_resampled = CatBoostClassifier(
            iterations=300, depth=8, learning_rate=0.1,
            random_state=42, verbose=False
        )
        cat_resampled.fit(X_resampled, y_resampled)

        # Evaluate
        pred = cat_resampled.predict(X_val)
        acc = accuracy_score(y_val, pred)
        results[name] = acc
        print(f"{name} Validation Accuracy: {acc:.4f}")
        print(f"Resampled shape: {X_resampled.shape}")
        print()

    except Exception as e:
        print(f"{name} failed: {e}\n")

# Find best oversampling technique
best_sampler = max(results, key=results.get)
print(f"Best oversampling technique: {best_sampler} with accuracy {results[best_sampler]:.4f}")

# Use best sampler for final training
print(f"\nUsing {best_sampler} for final training...")
best_sampler_instance = oversamplers[best_sampler]
X_train_best, y_train_best = best_sampler_instance.fit_resample(X_train, y_train)

# Train multiple models on best resampled data
print("\nTraining models on best resampled data...")
models_resampled = {
    'catboost': CatBoostClassifier(iterations=300, depth=8, learning_rate=0.1, random_state=42, verbose=False),
    'lgbm': LGBMClassifier(n_estimators=300, max_depth=8, learning_rate=0.1, random_state=42, verbose=-1),
    'xgboost': XGBClassifier(n_estimators=300, max_depth=8, learning_rate=0.1, random_state=42, use_label_encoder=False, eval_metric='mlogloss')
}

resampled_results = {}
for name, model in models_resampled.items():
    model.fit(X_train_best, y_train_best)
    pred = model.predict(X_val)
    acc = accuracy_score(y_val, pred)
    resampled_results[name] = acc
    print(f"{name} with {best_sampler}: {acc:.4f}")

print(f"\nBest model with {best_sampler}: {max(resampled_results, key=resampled_results.get)} with accuracy {max(resampled_results.values()):.4f}")

=== Data Augmentation dengan SMOTE Variants dan ADASYN ===

Testing different oversampling techniques...

Applying SMOTE...
SMOTE Validation Accuracy: 0.5437
Resampled shape: (2600, 102)

Applying ADASYN...
ADASYN failed: No samples will be generated with the provided ratio settings.

Applying BorderlineSMOTE...
BorderlineSMOTE Validation Accuracy: 0.5359
Resampled shape: (2600, 102)

Applying SVMSMOTE...
SVMSMOTE Validation Accuracy: 0.5125
Resampled shape: (2600, 102)

Applying SMOTETomek...
SMOTETomek Validation Accuracy: 0.5016
Resampled shape: (1788, 102)

Applying SMOTEENN...
SMOTEENN Validation Accuracy: 0.3344
Resampled shape: (76, 102)

Best oversampling technique: SMOTE with accuracy 0.5437

Using SMOTE for final training...

Training models on best resampled data...
catboost with SMOTE: 0.5437
lgbm with SMOTE: 0.5141
xgboost with SMOTE: 0.4969

Best model with SMOTE: catboost with accuracy 0.5437


In [23]:
print("=== Time-Series Models dengan LSTM/GRU ===\n")

import keras
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import numpy as np # Ensure numpy is imported
from sklearn.metrics import accuracy_score # Ensure accuracy_score is imported

# Prepare time-series data from nilai_minggu and aktivitas_hari columns
# Use X_train_smote to get column names, as it contains the relevant features
nilai_cols = [col for col in X_train_smote.columns if 'nilai_minggu' in col]
aktivitas_cols = [col for col in X_train_smote.columns if 'aktivitas_hari' in col]

# Extract time-series features
def prepare_ts_data(df, cols):
    """Prepare time-series data for LSTM/GRU"""
    ts_data = df[cols].values
    return ts_data

# Prepare sequences
def create_sequences(data, seq_length=4):
    """Create sequences for LSTM/GRU"""
    sequences = []
    for i in range(len(data) - seq_length + 1):
        sequences.append(data[i:i+seq_length])
    return np.array(sequences)

# Prepare data for training (using X_train_smote)
train_nilai_ts = prepare_ts_data(X_train_smote, nilai_cols)
train_aktivitas_ts = prepare_ts_data(X_train_smote, aktivitas_cols)

# Prepare data for validation (using X_val)
val_nilai_ts = prepare_ts_data(X_val, nilai_cols)
val_aktivitas_ts = prepare_ts_data(X_val, aktivitas_cols)

# Prepare data for testing (using X_test)
test_nilai_ts = prepare_ts_data(X_test, nilai_cols)
test_aktivitas_ts = prepare_ts_data(X_test, aktivitas_cols)


# Create sequences
seq_length = 4
train_nilai_seq = create_sequences(train_nilai_ts, seq_length)
train_aktivitas_seq = create_sequences(train_aktivitas_ts, seq_length)

val_nilai_seq = create_sequences(val_nilai_ts, seq_length)
val_aktivitas_seq = create_sequences(val_aktivitas_ts, seq_length)

test_nilai_seq = create_sequences(test_nilai_ts, seq_length)
test_aktivitas_seq = create_sequences(test_aktivitas_ts, seq_length)

# Adjust labels for sequences to match the length of the created sequences
y_train_seq_raw = y_train_smote.iloc[seq_length-1:] # Use .iloc for pandas Series
y_val_seq_raw = y_val.iloc[seq_length-1:] # Use .iloc for pandas Series

print(f"Train nilai sequences shape: {train_nilai_seq.shape}")
print(f"Train aktivitas sequences shape: {train_aktivitas_seq.shape}")
print(f"Validation nilai sequences shape: {val_nilai_seq.shape}")
print(f"Validation aktivitas sequences shape: {val_aktivitas_seq.shape}")
print(f"Test nilai sequences shape: {test_nilai_seq.shape}")

# Scale time-series data
from sklearn.preprocessing import StandardScaler, LabelEncoder
scaler_nilai = StandardScaler()
scaler_aktivitas = StandardScaler()

# Flatten the 3D sequences to 2D for scaling, then reshape back to 3D
train_nilai_scaled = scaler_nilai.fit_transform(train_nilai_seq.reshape(-1, train_nilai_seq.shape[-1])).reshape(train_nilai_seq.shape)
train_aktivitas_scaled = scaler_aktivitas.fit_transform(train_aktivitas_seq.reshape(-1, train_aktivitas_seq.shape[-1])).reshape(train_aktivitas_seq.shape)

val_nilai_scaled = scaler_nilai.transform(val_nilai_seq.reshape(-1, val_nilai_seq.shape[-1])).reshape(val_nilai_seq.shape)
val_aktivitas_scaled = scaler_aktivitas.transform(val_aktivitas_seq.reshape(-1, val_aktivitas_seq.shape[-1])).reshape(val_aktivitas_seq.shape)

test_nilai_scaled = scaler_nilai.transform(test_nilai_seq.reshape(-1, test_nilai_seq.shape[-1])).reshape(test_nilai_seq.shape)
test_aktivitas_scaled = scaler_aktivitas.transform(test_aktivitas_seq.reshape(-1, test_aktivitas_seq.shape[-1])).reshape(test_aktivitas_seq.shape)

# Encode labels
le_ts = LabelEncoder()
y_train_seq_encoded = le_ts.fit_transform(y_train_seq_raw)
y_val_seq_encoded = le_ts.transform(y_val_seq_raw)

# Define EarlyStopping and ReduceLROnPlateau here as they are used in training
early_stopping = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6)

# LSTM Model
def create_lstm_model(input_shape, num_classes):
    model = keras.Sequential([
        layers.Input(shape=input_shape),
        layers.LSTM(128, return_sequences=True),
        layers.Dropout(0.3),
        layers.LSTM(64, return_sequences=True),
        layers.Dropout(0.3),
        layers.LSTM(32),
        layers.Dropout(0.2),
        layers.Dense(16, activation='relu'),
        layers.Dense(num_classes, activation='softmax')
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# Train LSTM on nilai data
print("\nTraining LSTM on nilai data...")
lstm_nilai = create_lstm_model((seq_length, len(nilai_cols)), len(le_ts.classes_))
history_nilai = lstm_nilai.fit(
    train_nilai_scaled, y_train_seq_encoded,
    validation_data=(val_nilai_scaled, y_val_seq_encoded), # Use corrected validation data
    epochs=100,
    batch_size=32,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

# Train LSTM on aktivitas data
print("\nTraining LSTM on aktivitas data...")
lstm_aktivitas = create_lstm_model((seq_length, len(aktivitas_cols)), len(le_ts.classes_))
history_aktivitas = lstm_aktivitas.fit(
    train_aktivitas_scaled, y_train_seq_encoded,
    validation_data=(val_aktivitas_scaled, y_val_seq_encoded), # Use corrected validation data
    epochs=100,
    batch_size=32,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

# GRU Model
def create_gru_model(input_shape, num_classes):
    model = keras.Sequential([
        layers.Input(shape=input_shape),
        layers.GRU(128, return_sequences=True),
        layers.Dropout(0.3),
        layers.GRU(64, return_sequences=True),
        layers.Dropout(0.3),
        layers.GRU(32),
        layers.Dropout(0.2),
        layers.Dense(16, activation='relu'),
        layers.Dense(num_classes, activation='softmax')
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# Train GRU on combined data
print("\nTraining GRU on combined data...")
train_combined = np.concatenate([train_nilai_scaled, train_aktivitas_scaled], axis=2)
val_combined = np.concatenate([val_nilai_scaled, val_aktivitas_scaled], axis=2) # Create combined validation data
gru_combined = create_gru_model((seq_length, train_combined.shape[2]), len(le_ts.classes_))
history_combined = gru_combined.fit(
    train_combined, y_train_seq_encoded,
    validation_data=(val_combined, y_val_seq_encoded), # Use corrected validation data
    epochs=100,
    batch_size=32,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

# Evaluate models
print("\n=== Time-Series Model Evaluation ===")
lstm_nilai_pred = lstm_nilai.predict(val_nilai_scaled) # Predict on validation data
lstm_nilai_pred_class = np.argmax(lstm_nilai_pred, axis=1)
lstm_nilai_acc = accuracy_score(y_val_seq_encoded, lstm_nilai_pred_class)
print(f"LSTM (nilai) Validation Accuracy: {lstm_nilai_acc:.4f}")

lstm_aktivitas_pred = lstm_aktivitas.predict(val_aktivitas_scaled) # Predict on validation data
lstm_aktivitas_pred_class = np.argmax(lstm_aktivitas_pred, axis=1)
lstm_aktivitas_acc = accuracy_score(y_val_seq_encoded, lstm_aktivitas_pred_class)
print(f"LSTM (aktivitas) Validation Accuracy: {lstm_aktivitas_acc:.4f}")

gru_combined_pred = gru_combined.predict(val_combined) # Predict on validation data
gru_combined_pred_class = np.argmax(gru_combined_pred, axis=1)
gru_combined_acc = accuracy_score(y_val_seq_encoded, gru_combined_pred_class)
print(f"GRU (combined) Validation Accuracy: {gru_combined_acc:.4f}")

# Blend time-series models
print("\n=== Blending Time-Series Models ===")
ts_blend_proba = (lstm_nilai_pred + lstm_aktivitas_pred + gru_combined_pred) / 3
ts_blend_pred = np.argmax(ts_blend_proba, axis=1)
ts_blend_acc = accuracy_score(y_val_seq_encoded, ts_blend_pred)
print(f"Time-Series Blend Validation Accuracy: {ts_blend_acc:.4f}")

=== Time-Series Models dengan LSTM/GRU ===

Train nilai sequences shape: (2597, 4, 12)
Train aktivitas sequences shape: (2597, 4, 16)
Validation nilai sequences shape: (637, 4, 12)
Validation aktivitas sequences shape: (637, 4, 16)
Test nilai sequences shape: (797, 4, 12)

Training LSTM on nilai data...
Epoch 1/100
82/82 ━━━━━━━━━━━━━━━━━━━━ 10s 43ms/step - accuracy: 0.2487 - loss: 1.3874 - val_accuracy: 0.2732 - val_loss: 1.3861 - learning_rate: 0.0010
Epoch 2/100
82/82 ━━━━━━━━━━━━━━━━━━━━ 2s 28ms/step - accuracy: 0.2611 - loss: 1.3855 - val_accuracy: 0.2779 - val_loss: 1.3841 - learning_rate: 0.0010
Epoch 3/100
82/82 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.2591 - loss: 1.3847 - val_accuracy: 0.2543 - val_loss: 1.3890 - learning_rate: 0.0010
Epoch 4/100
82/82 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.2873 - loss: 1.3832 - val_accuracy: 0.2873 - val_loss: 1.3865 - learning_rate: 0.0010
Epoch 5/100
82/82 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.2869 - loss: 1.3804 

In [24]:
print("=== Generate Final Submission dengan Best Models ===\n")

# Apply best oversampling technique (from data augmentation cell)
# Using the best sampler found in the data augmentation experiments
best_sampler = SMOTE(random_state=42)  # Default, will be replaced with best from experiments
X_full_smote, y_full_smote = best_sampler.fit_resample(X, y)

print(f"Full training data shape after SMOTE: {X_full_smote.shape}")

# Train multiple best models on full dataset
print("\n=== Training Best Models on Full Dataset ===")

# 1. CatBoost (Optuna optimized if available, otherwise default)
print("Training CatBoost...")
cat_final = CatBoostClassifier(
    iterations=500, depth=8, learning_rate=0.1,
    random_state=42, verbose=False
)
cat_final.fit(X_full_smote, y_full_smote)

# 2. LightGBM
print("Training LightGBM...")
lgbm_final = LGBMClassifier(
    n_estimators=500, max_depth=10, learning_rate=0.08,
    random_state=42, verbose=-1
)
lgbm_final.fit(X_full_smote, y_full_smote)

# 3. XGBoost
print("Training XGBoost...")
xgb_final = XGBClassifier(
    n_estimators=500, max_depth=8, learning_rate=0.08,
    random_state=42, use_label_encoder=False, eval_metric='mlogloss'
)
xgb_final.fit(X_full_smote, y_full_smote)

# 4. Random Forest
print("Training Random Forest...")
rf_final = RandomForestClassifier(
    n_estimators=500, max_depth=12, random_state=42, n_jobs=-1
)
rf_final.fit(X_full_smote, y_full_smote)

# Get probabilities from all models
print("\n=== Getting Predictions ===")
cat_proba = cat_final.predict_proba(X_test)
lgbm_proba = lgbm_final.predict_proba(X_test)
xgb_proba = xgb_final.predict_proba(X_test)
rf_proba = rf_final.predict_proba(X_test)

# Advanced blending with optimized weights
# Based on validation results, use weights that gave best performance
weights = {
    'catboost': 0.4,
    'lgbm': 0.25,
    'xgboost': 0.2,
    'rf': 0.15
}

print("Blending predictions with optimized weights...")
blended_proba = (
    weights['catboost'] * cat_proba +
    weights['lgbm'] * lgbm_proba +
    weights['xgboost'] * xgb_proba +
    weights['rf'] * rf_proba
)

test_predictions = np.argmax(blended_proba, axis=1)

# Create submission
submission = pd.DataFrame({
    'id': test['id'],
    'target': test_predictions
})

# Save to Google Drive
submission.to_csv('/content/drive/MyDrive/Colab_Data/outputs/submission_final.csv', index=False)
print("Submission saved to: /content/drive/MyDrive/Colab_Data/outputs/submission_final.csv")

# Also save individual predictions for comparison
submission['target_catboost'] = np.argmax(cat_proba, axis=1)
submission['target_lgbm'] = np.argmax(lgbm_proba, axis=1)
submission['target_xgboost'] = np.argmax(xgb_proba, axis=1)
submission['target_rf'] = np.argmax(rf_proba, axis=1)
submission.to_csv('/content/drive/MyDrive/Colab_Data/outputs/submission_all_strategies.csv', index=False)
print("All strategies saved to: /content/drive/MyDrive/Colab_Data/outputs/submission_all_strategies.csv")

print(f"\n=== Submission Summary ===")
print(f"Submission shape: {submission.shape}")
print(f"\nBlending weights:")
for model, weight in weights.items():
    print(f"  {model}: {weight}")
print(f"\nFirst 5 predictions:")
print(submission.head())

print(f"\nTarget distribution in submission:")
print(submission['target'].value_counts().sort_index())

=== Generate Final Submission dengan Best Models ===

Full training data shape after SMOTE: (3252, 102)

=== Training Best Models on Full Dataset ===
Training CatBoost...
Training LightGBM...
Training XGBoost...
Training Random Forest...

=== Getting Predictions ===
Blending predictions with optimized weights...
Submission saved to: /content/drive/MyDrive/Colab_Data/outputs/submission_final.csv
All strategies saved to: /content/drive/MyDrive/Colab_Data/outputs/submission_all_strategies.csv

=== Submission Summary ===
Submission shape: (800, 6)

Blending weights:
  catboost: 0.4
  lgbm: 0.25
  xgboost: 0.2
  rf: 0.15

First 5 predictions:
   id  target  target_catboost  target_lgbm  target_xgboost  target_rf
0   3       0                0            2               0          2
1  12       2                2            2               2          2
2  14       2                3            2               1          3
3  18       3                3            3               3          3

In [25]:
print("=== Deep Learning dengan TensorFlow/Keras ===\n")

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import LabelEncoder

# Encode labels
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train_smote)
y_val_encoded = le.transform(y_val)

# Scale features
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_smote)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"Training shape: {X_train_scaled.shape}")
print(f"Validation shape: {X_val_scaled.shape}")
print(f"Number of classes: {len(le.classes_)}")

# Define deep learning model
def create_deep_model(input_dim, num_classes):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(64, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.2),
        layers.Dense(32, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(num_classes, activation='softmax')
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# Create and train model
print("\nTraining Deep Learning Model...")
dl_model = create_deep_model(X_train_scaled.shape[1], len(le.classes_))

# Callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6)

# Train
history = dl_model.fit(
    X_train_scaled, y_train_encoded,
    validation_data=(X_val_scaled, y_val_encoded),
    epochs=200,
    batch_size=32,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

# Evaluate
dl_val_pred_proba = dl_model.predict(X_val_scaled)
dl_val_pred = np.argmax(dl_val_pred_proba, axis=1)
dl_val_acc = accuracy_score(y_val_encoded, dl_val_pred)
print(f"\nDeep Learning Validation Accuracy: {dl_val_acc:.4f}")

# Compare with best traditional model
print(f"Best Traditional Model (CatBoost): {cat_acc:.4f}")
print(f"Improvement: {(dl_val_acc - cat_acc)*100:.2f}%")

=== Deep Learning dengan TensorFlow/Keras ===

Training shape: (2600, 102)
Validation shape: (640, 102)
Number of classes: 4

Training Deep Learning Model...
Epoch 1/200
82/82 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - accuracy: 0.3077 - loss: 1.6223 - val_accuracy: 0.4344 - val_loss: 1.2476 - learning_rate: 0.0010
Epoch 2/200
82/82 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.3950 - loss: 1.3442 - val_accuracy: 0.4609 - val_loss: 1.1656 - learning_rate: 0.0010
Epoch 3/200
82/82 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.4350 - loss: 1.2373 - val_accuracy: 0.4922 - val_loss: 1.1047 - learning_rate: 0.0010
Epoch 4/200
82/82 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.4562 - loss: 1.1745 - val_accuracy: 0.4938 - val_loss: 1.0696 - learning_rate: 0.0010
Epoch 5/200
82/82 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.4738 - loss: 1.1258 - val_accuracy: 0.5000 - val_loss: 1.0433 - learning_rate: 0.0010
Epoch 6/200
82/82 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.4946 - loss: 1.0981

In [26]:
print("=== Advanced Ensemble with Multiple Models ===\n")

from sklearn.ensemble import ExtraTreesClassifier, AdaBoostClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis

# Train multiple diverse models
models = {
    'catboost': CatBoostClassifier(iterations=436, depth=5, learning_rate=0.16,
                                   l2_leaf_reg=11.73, border_count=228, random_state=42, verbose=False),
    'lgbm': LGBMClassifier(n_estimators=400, max_depth=10, learning_rate=0.08, random_state=42, verbose=-1),
    'xgboost': XGBClassifier(n_estimators=400, max_depth=8, learning_rate=0.08, random_state=42,
                            use_label_encoder=False, eval_metric='mlogloss'),
    'rf': RandomForestClassifier(n_estimators=400, max_depth=12, random_state=42, n_jobs=-1),
    'et': ExtraTreesClassifier(n_estimators=400, max_depth=12, random_state=42, n_jobs=-1),
    'gb': GradientBoostingClassifier(n_estimators=400, max_depth=8, learning_rate=0.08, random_state=42)
}

model_predictions = {}
model_accuracies = {}

print("Training individual models...")
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train_smote, y_train_smote)
    pred = model.predict(X_val)
    acc = accuracy_score(y_val, pred)
    model_predictions[name] = pred
    model_accuracies[name] = acc
    print(f"{name} Validation Accuracy: {acc:.4f}")

print(f"\nBest individual model: {max(model_accuracies, key=model_accuracies.get)} with accuracy {max(model_accuracies.values()):.4f}")

# Advanced blending with top 3 models
top_models = sorted(model_accuracies.items(), key=lambda x: x[1], reverse=True)[:3]
print(f"\nTop 3 models: {top_models}")

# Get probabilities from top models
top_model_names = [name for name, _ in top_models]
model_probas = {}

for name in top_model_names:
    model_probas[name] = models[name].predict_proba(X_val)

# Try different weight combinations for top 3 models
print("\n=== Advanced Blending with Top 3 Models ===")
best_blend_acc = 0
best_weights = None

# Systematic weight search
weight_combinations = [
    (0.5, 0.3, 0.2),
    (0.6, 0.25, 0.15),
    (0.7, 0.2, 0.1),
    (0.4, 0.4, 0.2),
    (0.45, 0.35, 0.2),
    (0.55, 0.3, 0.15),
    (0.65, 0.25, 0.1),
    (0.5, 0.4, 0.1),
    (0.6, 0.3, 0.1),
]

for w1, w2, w3 in weight_combinations:
    blended_proba = (w1 * model_probas[top_model_names[0]] +
                    w2 * model_probas[top_model_names[1]] +
                    w3 * model_probas[top_model_names[2]])
    blended_pred = np.argmax(blended_proba, axis=1)
    blended_acc = accuracy_score(y_val, blended_pred)
    print(f"Blending ({top_model_names[0]}={w1}, {top_model_names[1]}={w2}, {top_model_names[2]}={w3}): {blended_acc:.4f}")

    if blended_acc > best_blend_acc:
        best_blend_acc = blended_acc
        best_weights = (w1, w2, w3)

print(f"\nBest 3-model blending: {best_blend_acc:.4f} with weights {best_weights}")

=== Advanced Ensemble with Multiple Models ===

Training individual models...
Training catboost...
catboost Validation Accuracy: 0.5391
Training lgbm...
lgbm Validation Accuracy: 0.5094
Training xgboost...
xgboost Validation Accuracy: 0.4984
Training rf...
rf Validation Accuracy: 0.4641
Training et...
et Validation Accuracy: 0.4516
Training gb...
gb Validation Accuracy: 0.4938

Best individual model: catboost with accuracy 0.5391

Top 3 models: [('catboost', 0.5390625), ('lgbm', 0.509375), ('xgboost', 0.4984375)]

=== Advanced Blending with Top 3 Models ===
Blending (catboost=0.5, lgbm=0.3, xgboost=0.2): 0.5266
Blending (catboost=0.6, lgbm=0.25, xgboost=0.15): 0.5406
Blending (catboost=0.7, lgbm=0.2, xgboost=0.1): 0.5453
Blending (catboost=0.4, lgbm=0.4, xgboost=0.2): 0.5188
Blending (catboost=0.45, lgbm=0.35, xgboost=0.2): 0.5156
Blending (catboost=0.55, lgbm=0.3, xgboost=0.15): 0.5391
Blending (catboost=0.65, lgbm=0.25, xgboost=0.1): 0.5453
Blending (catboost=0.5, lgbm=0.4, xgboost=0

In [27]:
print("=== Generate Final Submission with Advanced Ensemble ===\n")

# Apply SMOTE pada full training data
smote_full = SMOTE(random_state=42)
X_full_smote, y_full_smote = smote_full.fit_resample(X, y)

# Select top features based on mutual information
mi_scores = mutual_info_classif(X_full_smote, y_full_smote)
mi_df = pd.DataFrame({'feature': X_full_smote.columns, 'mi_score': mi_scores})
mi_df = mi_df.sort_values('mi_score', ascending=False)
top_n = 60
top_features = mi_df.head(top_n)['feature'].tolist()

X_full_selected = X_full_smote[top_features]
X_test_selected = X_test[top_features]

print(f"Using top {top_n} features for final training")

# Train multiple models on selected features
print("\nTraining CatBoost on selected features...")
cat_final = CatBoostClassifier(
    iterations=500, depth=6, learning_rate=0.12,
    l2_leaf_reg=10, border_count=200, random_state=42, verbose=False
)
cat_final.fit(X_full_selected, y_full_smote)

print("Training LightGBM on selected features...")
lgbm_final = LGBMClassifier(n_estimators=500, max_depth=10, learning_rate=0.08, random_state=42, verbose=-1)
lgbm_final.fit(X_full_selected, y_full_smote)

print("Training XGBoost on selected features...")
xgb_final = XGBClassifier(n_estimators=500, max_depth=8, learning_rate=0.08, random_state=42,
                        use_label_encoder=False, eval_metric='mlogloss')
xgb_final.fit(X_full_selected, y_full_smote)

# Get probabilities for test set
cat_proba_test = cat_final.predict_proba(X_test_selected)
lgbm_proba_test = lgbm_final.predict_proba(X_test_selected)
xgb_proba_test = xgb_final.predict_proba(X_test_selected)

# Advanced blending with optimized weights (based on validation results)
best_w_cat, best_w_lgbm, best_w_xgb = 0.5, 0.3, 0.2

# Blending
blended_proba_test = (best_w_cat * cat_proba_test +
                     best_w_lgbm * lgbm_proba_test +
                     best_w_xgb * xgb_proba_test)
test_predictions = np.argmax(blended_proba_test, axis=1)

# Buat submission file
submission = pd.DataFrame({
    'id': test['id'],
    'target': test_predictions
})

submission.to_csv('/content/drive/MyDrive/Colab_Data/outputs/submission_final.csv', index=False)
print("Submission file saved as '/content/drive/MyDrive/Colab_Data/outputs/submission_final.csv'")
print(f"\nSubmission shape: {submission.shape}")
print(submission.head())

# Simpan juga individual predictions untuk comparison
submission['target_catboost'] = np.argmax(cat_proba_test, axis=1)
submission['target_lgbm'] = np.argmax(lgbm_proba_test, axis=1)
submission['target_xgboost'] = np.argmax(xgb_proba_test, axis=1)
submission.to_csv('/content/drive/MyDrive/Colab_Data/outputs/submission_all_strategies.csv', index=False)
print("\nAll strategy predictions saved as '/content/drive/MyDrive/Colab_Data/outputs/submission_all_strategies.csv'")

print(f"\n=== Summary ===")
print(f"Submission strategy: Advanced Blending (CatBoost={best_w_cat}, LightGBM={best_w_lgbm}, XGBoost={best_w_xgb})")
print(f"Features used: {top_n} (selected by Mutual Information)")
print(f"Files generated:")
print("1. /content/drive/MyDrive/Colab_Data/outputs/submission_final.csv - Advanced blending (recommended)")
print("2. /content/drive/MyDrive/Colab_Data/outputs/submission_all_strategies.csv - Individual predictions for comparison")

=== Generate Final Submission with Advanced Ensemble ===

Using top 60 features for final training

Training CatBoost on selected features...
Training LightGBM on selected features...
Training XGBoost on selected features...
Submission file saved as '/content/drive/MyDrive/Colab_Data/outputs/submission_final.csv'

Submission shape: (800, 2)
   id  target
0   3       0
1  12       2
2  14       2
3  18       3
4  28       0

All strategy predictions saved as '/content/drive/MyDrive/Colab_Data/outputs/submission_all_strategies.csv'

=== Summary ===
Submission strategy: Advanced Blending (CatBoost=0.5, LightGBM=0.3, XGBoost=0.2)
Features used: 60 (selected by Mutual Information)
Files generated:
1. /content/drive/MyDrive/Colab_Data/outputs/submission_final.csv - Advanced blending (recommended)
2. /content/drive/MyDrive/Colab_Data/outputs/submission_all_strategies.csv - Individual predictions for comparison
